# SCICO — L-ADMM TV reconstruction (cone-beam, sparse views)

**Algorithm:** Linearised ADMM (L-ADMM) for the problem
$$\min_{x\ge 0}\;\frac{1}{2}\|Ax-b\|^2 + \lambda\,\mathrm{TV}(x)$$

* **Forward / adjoint** — TIGRE cone-beam GPU projectors (`Ax` / `Atb`)
* **TV proximal** — SCICO `FiniteDifference` + `L21Norm.prox` (block-wise soft-threshold, exact)
* **Iterations** (per outer step)
  1. *x-update* — gradient step on the augmented Lagrangian: $x \leftarrow \mathrm{clip}\bigl(x - \tau(A^\top(Ax-b) + \rho\,D^\top(Dx-z+u)),\;0,\infty\bigr)$
  2. *z-update* — TV proximal: $z \leftarrow \mathrm{prox}_{(\lambda/\rho)\|\cdot\|_{2,1}}(Dx+u)$
  3. *u-update* — dual ascent: $u \leftarrow u + Dx - z$

Step size $\tau = 1/(L + \rho\|D\|^2)$ guarantees convergence.

**Input:** `calibrated_radiographs.npz` — already flat/dark-corrected and $-\log$ transformed, shape `(360,700,400)`, sparse subset every `SPARSE_STEP` views.

In [ ]:
import numpy as np
import jax.numpy as jnp
import tigre
import tigre.algorithms as algs
import nibabel as nib
import matplotlib.pyplot as plt
from time import time

from scico import functional, linop
from scico.util import device_info

print(device_info())
print(f'TIGRE: {tigre.__file__}')

## Parameters — adapt here

In [ ]:
# ── data ──────────────────────────────────────────────────────────────────────
DATA_PATH   = '~/calibrated_radiographs.npz'
SPARSE_STEP = 10          # every 10 views  →  36 views @ 10° spacing

# ── cone-beam geometry ────────────────────────────────────────────────────────
SDD        = 557.0        # mm  source-to-detector distance
SOD        = 530.0        # mm  source-to-object distance
PIXEL_SIZE = 0.117        # mm  detector pixel size

# ── reconstruction grid ───────────────────────────────────────────────────────
# Full resolution: N_XY=400, N_Z=700.  Coarser grids run faster for testing.
N_XY = 400    # transaxial side length  [voxels]
N_Z  = 700    # axial height            [voxels]

# ── L-ADMM hyperparameters ────────────────────────────────────────────────────
LAM       = 0.005   # TV regularisation weight λ
RHO       = 1.0     # augmented-Lagrangian penalty ρ
N_ITER    = 150     # outer L-ADMM iterations
N_POWER   = 8       # power iterations to estimate ||A||_2^2
D_NORM_SQ = 12.0    # safe upper bound on spectral norm of D^T D (3-D forward diff)

## Load sparse projections

In [ ]:
data     = np.load(DATA_PATH)
all_proj = data['data']                               # (360, 700, 400) float32
proj     = all_proj[::SPARSE_STEP].astype(np.float32) # (36, 700, 400)

n_angles, n_det_v, n_det_u = proj.shape
angles = np.linspace(0, 2 * np.pi, n_angles, endpoint=False).astype(np.float32)

print(f'Sparse projections : {proj.shape}   dtype={proj.dtype}')
print(f'Value range        : [{proj.min():.4f}, {proj.max():.4f}]')
print(f'Angles             : {n_angles} views, step {np.degrees(angles[1]):.1f}°')

## TIGRE cone-beam geometry

TIGRE convention: `nVoxel = [nX, nZ, nY]` (Z = axial/height), NumPy volume shape `(nX, nZ, nY)`.

In [ ]:
# Voxel size at iso-centre, then scaled for the chosen grid resolution
voxel_size_full = PIXEL_SIZE * SOD / SDD   # ≈ 0.1113 mm at full 400-voxel grid
scale           = 400 / N_XY              # downsampling factor
dv              = voxel_size_full * scale  # effective voxel size [mm]

geo = tigre.geometry()
geo.mode = 'cone'
geo.DSD  = SDD;  geo.DSO = SOD

geo.nDetector = np.array([n_det_v, n_det_u])
geo.dDetector = np.array([PIXEL_SIZE, PIXEL_SIZE])
geo.sDetector = geo.nDetector * geo.dDetector

geo.nVoxel    = np.array([N_XY, N_XY, N_Z])
geo.dVoxel    = np.array([dv, dv, dv])
geo.sVoxel    = geo.nVoxel * geo.dVoxel

geo.offOrigin   = np.zeros(3)
geo.offDetector = np.zeros(2)
geo.COR         = 0.0
geo.rotDetector = np.zeros(3)
geo.accuracy    = 0.5

print(geo)
print(f'Reconstruction grid : {tuple(geo.nVoxel)}  (nX, nZ, nY)')
print(f'Voxel size          : {dv:.4f} mm')

## Verify projectors

In [ ]:
dummy = np.zeros(tuple(geo.nVoxel), dtype=np.float32)
fwd   = tigre.Ax(dummy, geo, angles)
bwd   = tigre.Atb(fwd, geo, angles, 'matched')
print(f'Ax  : {dummy.shape}  →  {fwd.shape}   (n_angles, nDetV, nDetU)')
print(f'Atb : {fwd.shape}   →  {bwd.shape}   (nX, nZ, nY)')
del dummy, fwd, bwd

## Power iteration — estimate Lipschitz constant $L = \|A^\top A\|_2$

Step size $\tau = 1/(L + \rho\|D\|^2)$ guarantees L-ADMM convergence.

In [ ]:
rng_pi = np.random.default_rng(0)
x_pi   = rng_pi.standard_normal(tuple(geo.nVoxel)).astype(np.float32)
x_pi  /= np.linalg.norm(x_pi)

print('Power iteration ...')
t0 = time()
for k in range(N_POWER):
    Ax_pi = tigre.Ax(x_pi, geo, angles)
    x_pi  = tigre.Atb(Ax_pi, geo, angles, 'matched').astype(np.float32)
    L_est = float(np.linalg.norm(x_pi))
    x_pi /= L_est
    print(f'  [{k+1:2d}/{N_POWER}]  L ≈ {L_est:.4e}  ({time()-t0:.0f}s)')

L   = L_est
tau = 1.0 / (L + RHO * D_NORM_SQ)
del x_pi, Ax_pi
print(f'\nL = {L:.4e}   τ = {tau:.4e}')

## FDK warm start

In [ ]:
print('Running FDK for warm start ...')
t0    = time()
x_fdk = algs.fdk(proj, geo, angles).astype(np.float32)
x_fdk = np.clip(x_fdk, 0.0, None)
print(f'FDK done in {time()-t0:.1f}s   range [{x_fdk.min():.4f}, {x_fdk.max():.4f}]')

## FDK quick look (warm-start quality)

In [ ]:
nX_f, nZ_f, nY_f = x_fdk.shape
vmax_f = float(np.percentile(x_fdk, 99.5))
kw_f   = dict(cmap='gray', origin='lower', vmin=0, vmax=vmax_f)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
# Axial: nX × nY plane  (rotation axis is the normal to this slice)
axes[0].imshow(x_fdk[:, nZ_f//2, :],   **kw_f);  axes[0].set_title(f'FDK — axial z={nZ_f//2}');      axes[0].axis('off')
# Sagittal: nZ (rows=vertical=rotation axis) × nY (cols=horizontal=transaxial)
axes[1].imshow(x_fdk[nX_f//2, :, :],   **kw_f);  axes[1].set_title(f'FDK — sagittal x={nX_f//2}');   axes[1].axis('off')
# Coronal: transpose so nZ (rotation axis) is rows/vertical, nX is cols/horizontal
axes[2].imshow(x_fdk[:, :, nY_f//2].T, **kw_f);  axes[2].set_title(f'FDK — coronal y={nY_f//2}');    axes[2].axis('off')
plt.suptitle(f'FDK warm start  |  {n_angles} views  grid {N_XY}×{N_Z}×{N_XY}')
plt.tight_layout();  plt.show()

## L-ADMM TV reconstruction  (SCICO finite differences + TIGRE projectors)

SCICO provides:
* `linop.FiniteDifference` — 3-D gradient operator $D$, output shape `(3, nX, nZ, nY)`
* `functional.L21Norm().prox` — exact block-wise soft-threshold (TV $z$-update)

TIGRE provides GPU cone-beam $A x$ and $A^\top y$.

In [ ]:
# ── SCICO operators ──────────────────────────────────────────────────────────
vol_shape = tuple(geo.nVoxel)            # (nX, nZ, nY)
D = linop.FiniteDifference(input_shape=vol_shape, append=0, circular=False)
g = functional.L21Norm()                 # prox = block-wise soft-threshold

lam = LAM
rho = RHO
# tau already computed above from power iteration
print(f'τ={tau:.4e}  λ={lam}  ρ={rho}  vol={vol_shape}')

# ── initialise variables ─────────────────────────────────────────────────────
x      = jnp.array(x_fdk)               # primal: reconstruction  (nX, nZ, nY)
z      = D(x)                            # auxiliary: Dx           (3, nX, nZ, nY)
u      = jnp.zeros_like(z)              # scaled dual
proj_j = jnp.array(proj)               # measurements  (n_angles, nDetV, nDetU)

# ── monitoring ───────────────────────────────────────────────────────────────
objectives, res_primal, res_dual = [], [], []
LOG_EVERY = 10

print(f'\nL-ADMM: {N_ITER} iterations')
print('-' * 72)
t_total = time()

for k in range(N_ITER):

    # ── (1) x-update: gradient step on augmented Lagrangian ──────────────────
    x_np      = np.asarray(x, dtype=np.float32)
    Ax_np     = tigre.Ax(x_np, geo, angles)                       # GPU
    res_np    = Ax_np - np.asarray(proj_j, dtype=np.float32)
    grad_data = jnp.array(tigre.Atb(res_np, geo, angles, 'matched'))  # GPU

    Dx        = D(x)
    grad_reg  = D.T(Dx - z + u)                                   # JAX CPU

    x_new = jnp.clip(x - tau * (grad_data + rho * grad_reg), 0.0, None)

    # ── (2) z-update: TV proximal via SCICO L21Norm.prox ─────────────────────
    z_old  = z
    Dx_new = D(x_new)
    z      = g.prox(Dx_new + u, lam / rho)                        # block soft-threshold

    # ── (3) u-update: dual ascent ─────────────────────────────────────────────
    u = u + Dx_new - z

    x = x_new

    # ── logging ───────────────────────────────────────────────────────────────
    if k % LOG_EVERY == 0:
        data_fid = 0.5 * float(jnp.sum((jnp.array(Ax_np) - proj_j) ** 2))
        tv_val   = float(g(Dx_new))
        obj      = data_fid + lam * tv_val
        r_p      = float(jnp.linalg.norm(Dx_new - z))
        r_d      = float(jnp.linalg.norm(rho * D.T(z - z_old)))
        objectives.append(obj)
        res_primal.append(r_p)
        res_dual.append(r_d)
        print(f'[{k:4d}/{N_ITER}]  obj={obj:.4e}  TV={tv_val:.4e}  '
              f'|r_p|={r_p:.3e}  |r_d|={r_d:.3e}  t={time()-t_total:.0f}s')

elapsed_total = time() - t_total
print(f'\nL-ADMM done in {elapsed_total:.0f}s  ({elapsed_total/60:.1f} min)')

recon = np.asarray(x, dtype=np.float32)
print(f'Reconstruction range: [{recon.min():.4f}, {recon.max():.4f}]')

## Convergence diagnostics

In [ ]:
iters_log = list(range(0, N_ITER, LOG_EVERY))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].semilogy(iters_log, objectives);  axes[0].set(title='Objective $f+\\lambda g$', xlabel='iteration')
axes[1].semilogy(iters_log, res_primal);  axes[1].set(title='Primal residual $\\|Dx-z\\|$', xlabel='iteration')
axes[2].semilogy(iters_log, res_dual);    axes[2].set(title='Dual residual', xlabel='iteration')
for ax in axes:
    ax.grid(True, which='both', ls=':')
plt.tight_layout();  plt.show()

## Visualise — FDK vs L-ADMM TV (three orthogonal slices)

In [ ]:
nX, nZ, nY = recon.shape
vmax = float(np.percentile(recon, 99.5))
kw   = dict(cmap='gray', origin='lower', vmin=0, vmax=vmax)
kw_f = dict(cmap='gray', origin='lower', vmin=0, vmax=float(np.percentile(x_fdk, 99.5)))

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# ── row 0: FDK ────────────────────────────────────────────────────────────────
# Axial: nX × nY  (rotation axis normal to slice)
axes[0, 0].imshow(x_fdk[:, nZ//2, :],   **kw_f);  axes[0, 0].set_title('FDK — axial');     axes[0, 0].axis('off')
# Sagittal: nZ (vertical=rotation axis) × nY (horizontal=transaxial)
axes[0, 1].imshow(x_fdk[nX//2, :, :],   **kw_f);  axes[0, 1].set_title('FDK — sagittal');  axes[0, 1].axis('off')
# Coronal: .T so nZ (rotation axis) is rows/vertical, nX is cols/horizontal
axes[0, 2].imshow(x_fdk[:, :, nY//2].T, **kw_f);  axes[0, 2].set_title('FDK — coronal');   axes[0, 2].axis('off')

# ── row 1: L-ADMM TV ──────────────────────────────────────────────────────────
axes[1, 0].imshow(recon[:, nZ//2, :],   **kw);  axes[1, 0].set_title('L-ADMM TV — axial');     axes[1, 0].axis('off')
axes[1, 1].imshow(recon[nX//2, :, :],   **kw);  axes[1, 1].set_title('L-ADMM TV — sagittal');  axes[1, 1].axis('off')
axes[1, 2].imshow(recon[:, :, nY//2].T, **kw);  axes[1, 2].set_title('L-ADMM TV — coronal');   axes[1, 2].axis('off')

plt.suptitle(
    f'SCICO L-ADMM TV  |  {n_angles} views (step {SPARSE_STEP})  '
    f'λ={lam}  ρ={rho}  iter={N_ITER}  grid {N_XY}×{N_Z}×{N_XY}',
    fontsize=11
)
plt.tight_layout()
out_png = f'recon_scico_ladmm_tv_{N_XY}x{N_Z}_lam{lam}.png'
plt.savefig(out_png, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved: {out_png}')

## Save NIfTI

In [ ]:
out_nii = f'recon_scico_ladmm_tv_{N_XY}x{N_Z}.nii'

# TIGRE shape (nX, nZ, nY) → NIfTI (nZ, nX, nY)
vol_nii = np.ascontiguousarray(np.transpose(recon, (1, 0, 2)), dtype=np.float32)

affine       = np.eye(4, dtype=np.float64)
affine[0, 0] = dv
affine[1, 1] = dv
affine[2, 2] = dv

nib.save(nib.Nifti1Image(vol_nii, affine), out_nii)
print(f'Saved: {out_nii}   shape={vol_nii.shape}   voxel={dv:.4f} mm')